# Week 9 — Monday: Preparing the Data — Scaling, Encoding and Pipelines

**DATA 202 · Calvin University**

April 1912: the RMS *Titanic* sank on its first voyage. Its passenger list survives: 1,309 passengers, with their ticket class, sex, age, family aboard, fare and port.

> 🎯 **Can kNN learn who survived from a list that mixes years, pounds and words?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~5 min | 1 · The passenger list |
| ~10 min | 2 · Years vs. pounds: scaling (SLO 09A) |
| ~10 min | 3 · Words to numbers: encoding (SLO 09A) |
| ~12 min | 4 · One pipe for every step: pipelines (SLO 09B) |
| ~3 min | 5 · Who did we leave out? |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

<sub>Data: *titanic3*, compiled by Thomas Cason from Encyclopedia Titanica; distributed by Frank Harrell, Vanderbilt Biostatistics</sub>

---
## 1 · The Passenger List · ~5 min

In [ ]:
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

titanic = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/titanic.csv")
titanic.head()

| column | means |
|---|---|
| `pclass` | ticket class: 1st, 2nd or 3rd |
| `sex` | female or male |
| `age` | in years |
| `sibsp` | siblings + spouses aboard |
| `parch` | parents + children aboard |
| `fare` | ticket price, in 1912 pounds (£) |
| `cabin` | cabin number |
| `embarked` | port: Southampton, Cherbourg or Queenstown |
| `survived` | **the label**: 1 = survived, 0 = died |

In [ ]:
passengers = titanic.dropna(subset=["age", "fare", "embarked"])     # today: passengers with these recorded
X = passengers[["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]]
y = passengers["survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=1)
print(len(passengers), "passengers |", len(X_train), "training |", len(X_test), "test")

Today: the 1,043 passengers whose age, fare and port were recorded. **Wednesday:** the 266 we just left out.

> 💬 **Question:** kNN measures distances between rows of numbers. Which columns can it use as they are?

<details><summary>Answer</summary>

* **Numbers:** `age`, `sibsp`, `parch`, `fare`, and `pclass`
* **Words:** `sex` and `embarked`. What is "female" minus "male"?
* Even the numbers aren't ready: years, people and pounds are different rulers (Part 2). And is `pclass` really a number (Part 3)?
</details>

---
## 2 · Years vs. Pounds: Scaling (SLO 09A) · ~10 min

In [ ]:
numeric = ["pclass", "age", "sibsp", "parch", "fare"]
fig = px.box(X_train[numeric], title="Five Numeric Columns, Five Different Rulers (782 training passengers)")
fig.show()

Fares run from £0 to £512. Ages stop at 80; family counts at 8.

> 📄 **Handout 1 — Who is closest?** Three training passengers. Is A closer to B or to C? First in years and pounds, then in standard deviations. 🗣️ Compare with a neighbor.

In [ ]:
names = {"A": "Colley, Mr. Edward Pomeroy", "B": "Davies, Mr. John Samuel", "C": "Chaffee, Mr. Herbert Fuller"}
three = passengers.set_index("name").loc[list(names.values()), ["pclass", "age", "fare"]]
three.index = list(names)
three.insert(0, "name", list(names.values()))
display(three)

sd = X_train[["age", "fare"]].std(ddof=0)                     # spread of each column, training passengers
for other in ["B", "C"]:
    d_age = three.loc["A", "age"] - three.loc[other, "age"]
    d_fare = three.loc["A", "fare"] - three.loc[other, "fare"]
    raw = (d_age ** 2 + d_fare ** 2) ** 0.5
    scaled = ((d_age / sd["age"]) ** 2 + (d_fare / sd["fare"]) ** 2) ** 0.5
    print(f"A to {other} | years and pounds: {raw:5.1f} | standard deviations: {scaled:.2f}")

<details><summary>Answer</summary>

* **Years and pounds:** B is closer (26.0 vs. 35.6). B is 26 years younger, but his fare is almost the same, and £1 counts as much as 1 year
* **Standard deviations** (age ÷ 14.5, fare ÷ 50.2): C is closer (0.71 vs. 1.79). C is A's age, and £36 is less than one standard deviation of fare
* Same three people, a different nearest neighbor. The **units** chose it
</details>

**`StandardScaler`**, one column at a time: z = (value − mean) ÷ standard deviation

* Every column ends up with mean 0 and standard deviation 1: one ruler for all
* The mean and standard deviation come from the **training** passengers only

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

for name, model in [("kNN, raw", KNeighborsClassifier(n_neighbors=5)),
                    ("kNN, scaled", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))),
                    ("tree, raw", DecisionTreeClassifier(max_depth=4, random_state=0)),
                    ("tree, scaled", make_pipeline(StandardScaler(), DecisionTreeClassifier(max_depth=4, random_state=0)))]:
    score = cross_val_score(model, X_train[numeric], y_train, cv=5).mean()
    print(f"{name:13} | 5-fold CV accuracy: {score:.3f}")

<details><summary>Answer</summary>

* **kNN:** 63.3% → 68.2% after scaling. It measures distances, so units matter
* **Tree:** 68.4% both ways. Each question looks at one column ("fare ≤ 52?"), so the units never meet
* Scale for models that measure distance (kNN, k-means, PCA)
</details>

> 📌 Cross-validated on the 782 training passengers. The test set stays locked.

---
## 3 · Words to Numbers: Encoding (SLO 09A) · ~10 min

**One-hot encoding:** one new column per category, with 1 if the passenger has it and 0 if not.

| embarked | → | embarked_Cherbourg | embarked_Queenstown | embarked_Southampton |
|---|---|:-:|:-:|:-:|
| Southampton | | 0 | 0 | 1 |
| Cherbourg | | 1 | 0 | 0 |

> 📄 **Handout 2 — Encode four passengers.** Fill in the 0s and 1s for `sex` and `embarked`.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(drop="if_binary", sparse_output=False).fit(X_train[["sex", "embarked"]])
four = X_train[["sex", "embarked"]].drop_duplicates().head(4)          # four different combinations
display(four)
pd.DataFrame(encoder.transform(four), columns=encoder.get_feature_names_out(), index=four.index).astype(int)

<details><summary>Answer</summary>

* `embarked` → 3 columns, with exactly one 1 in each row
* `sex` → **one** column, `sex_male`. A `sex_female` column would just repeat it backwards, so `drop="if_binary"` leaves it out
</details>

> 💬 **Question:** Why not code the ports as Southampton = 1, Cherbourg = 2, Queenstown = 3?

<details><summary>Answer</summary>

kNN would treat Queenstown as twice as far from Southampton as Cherbourg is. The ports have no such order; those numbers would invent one.
</details>

`pclass` is already 1, 2, 3. Is it a **rank** or a **name**?

> "From the point of view of social order, the most basic kinds of processes are those of nominalization and ordinalization: the naming of kinds and the designation of ranks. … In practice, ranks can be collapsed into types."
> — Fourcade & Healy, *The Ordinal Society*, ch. 3

> 💬 **Question:** Keep `pclass` as one ranked column (1 < 2 < 3), or one-hot it into three kinds? What does each choice tell kNN about the passengers?

<details><summary>Answer</summary>

* **Rank (one column):** 1st class sits closer to 2nd than to 3rd. That fits fares, decks and distance from the lifeboats
* **Kinds (three columns):** every class is equally far from every other
* Neither is "the data". It's a choice we make. Today we keep the ranks. For words with an order ("low", "medium", "high") there is `OrdinalEncoder`
</details>

---
## 4 · One Pipe for Every Step: Pipelines (SLO 09B) · ~12 min

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/09/images/pipeline.png" width="380" alt="A drawing of connected pipes: water goes in at one end, through many bends and joints, and out at the other.">

A **pipeline** joins the steps: each step's output flows into the next one. A **column transformer** sends different columns down different pipes.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric),                          # ("name", step, columns)
    ("cat", OneHotEncoder(drop="if_binary"), ["sex", "embarked"]),
])
model = Pipeline([
    ("prep", preprocess),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])
model                                                            # click the boxes to open them

> 📄 **Handout 3 — Follow the pipe.** Write which columns go into each branch, and how many columns come out of `prep` into kNN.

In [ ]:
model.fit(X_train, y_train)
print(model.named_steps["prep"].get_feature_names_out())
print("5-fold CV accuracy:", round(cross_val_score(model, X_train, y_train, cv=5).mean(), 3))

<details><summary>Answer</summary>

* 5 scaled numbers + `sex_male` + 3 ports = **9 columns** into kNN
* 68.2% → **76.1%**. The most useful column was a word: "women and children first"
</details>

**Why a pipeline?**

1. **One object:** `fit` and `predict` run every step, in order, on any new passenger
2. **No leaks:** the scaler learns its mean and standard deviation from the training rows only. Inside `cross_val_score`, it relearns them in every round, from the 4 folds it fits on. Practice 05's rule — test rows never go into `fit` — kept automatically
3. **Tune any step:** step name + `__` + knob, e.g. `knn__n_neighbors`

In [ ]:
search = GridSearchCV(model, {"knn__n_neighbors": [1, 3, 5, 9, 15, 25, 51, 101, 201]}, cv=5, return_train_score=True)
search.fit(X_train, y_train)

curve = pd.DataFrame(search.cv_results_)[["param_knn__n_neighbors", "mean_train_score", "mean_test_score"]]
curve.columns = ["k", "training folds", "validation fold"]
fig = px.line(curve, x="k", y=["training folds", "validation fold"], markers=True, log_x=True,
              labels={"value": "accuracy", "variable": "scored on"}, title="Choosing k for the Whole Pipeline")
fig.show()

> 📄 **Handout 4 — Choose k.** Circle the best k. Label the overfitting and underfitting sides.

<details><summary>Answer</summary>

* **Best: k = 25** (77.2%); k = 15 is right behind (77.0%)
* **k = 1:** 98.8% on the folds it learned from, 71.4% on the held-out fold → **overfitting**
* **k = 201:** about 70% on both → **underfitting**
* Every round refit the scaler and the encoder too: 9 values of k × 5 folds = 45 whole pipelines
</details>

Now, **once**, the locked test set:

In [ ]:
print("best k:", search.best_params_["knn__n_neighbors"], "| CV:", round(search.best_score_, 3),
      "| test:", round(search.score(X_test, y_test), 3))
print("lazy model, 'everyone died':", round((y_test == 0).mean(), 3))

> 📌 **One split:** 81.2% is on *these* 261 test passengers; cross-validation said 77.2%.

---
## 5 · Who Did We Leave Out? · ~3 min

We started with 1,309 passengers and kept 1,043. The model never saw the other **266**.

> 🎯 **Predict First (for Wednesday):** Were the 266 like everyone else? If not, who were they?

---
### Wrap-up

1. **Scale** columns measured on different rulers before measuring distances; trees don't need it
2. **One-hot encode** categories with no order; keep or build ranks only when the order is real, and say so
3. A **pipeline** chains the steps, fits them on training rows only, and lets `GridSearchCV` tune any of them
4. 📌 Every score: one dataset, one split, and only the passengers who were fully recorded

**Wednesday:** missing data — the 266.